<div style='text-align: center; background-color: #0d1117; color: white; padding: 30px; border-radius: 12px; font-family: Arial, sans-serif;'>

<h1 style='color: #58a6ff; margin-bottom: 5px;'>Internet de las Cosas (IoT)</h1>
<h2 style='color: #ffffff; margin-top: 0;'>Actividad 4 — Almacenamiento y Procesamiento de Datos IoT</h2>

<hr style='border-color: #58a6ff; margin: 20px 0;'/>

<table style='margin: 0 auto; font-size: 16px; color: #cccccc;'>
<tr><td style='padding: 6px 20px; text-align: right; color: #58a6ff;'><b>Estudiante:</b></td><td style='padding: 6px 20px; text-align: left;'>Ana María García Arias</td></tr>
<tr><td style='padding: 6px 20px; text-align: right; color: #58a6ff;'><b>Programa:</b></td><td style='padding: 6px 20px; text-align: left;'>Maestría en Inteligencia Artificial</td></tr>
<tr><td style='padding: 6px 20px; text-align: right; color: #58a6ff;'><b>Asignatura:</b></td><td style='padding: 6px 20px; text-align: left;'>Internet de las Cosas</td></tr>
<tr><td style='padding: 6px 20px; text-align: right; color: #58a6ff;'><b>Docente:</b></td><td style='padding: 6px 20px; text-align: left;'>Cristian Duney Bermúdez Quintero</td></tr>
<tr><td style='padding: 6px 20px; text-align: right; color: #58a6ff;'><b>Fecha:</b></td><td style='padding: 6px 20px; text-align: left;'>Mayo 2026</td></tr>
</table>

</div>

---

## Objetivo de la actividad

Diseñar e implementar una infraestructura de almacenamiento y procesamiento de datos IoT que contemple:

1. **Selección de la infraestructura** de almacenamiento adecuada para series de tiempo.
2. **Implementación de la recepción** de datos desde el nodo sensor virtual.
3. **Técnicas de procesamiento**: preprocesamiento, filtrado y transformación.

---

## 1. Arquitectura implementada

```
CounterFit (DHT11 virtual)
        |
        v
nodo_timescale.py
        |   INSERT cada 30 s
        v
TimescaleDB Cloud (Tiger Cloud)
  tabla: lecturas_iot (hypertable)
        |
        +------------> Procesamiento_Datos_IoT.ipynb
        |              (preprocessing, filtrado, transformacion)
        |
        +------------> dashboard_iot.py (Streamlit)
                       http://localhost:8501
```

### ¿Por qué TimescaleDB?

| Criterio | SQLite | TimescaleDB Cloud |
|----------|--------|------------------|
| Tipo | Relacional local | Serie de tiempo cloud |
| Escalabilidad | Archivo único | Millones de filas/día |
| Funciones temporales | Limitadas | `time_bucket()`, `first()`, `last()` |
| Acceso remoto | No | Si (SSL/TLS) |
| Dashboard en vivo | Complejo | Nativo con SQL |
| Ideal para | Prototipado local | Producción IoT |

---

## 2. Esquema de la hypertable

```sql
CREATE TABLE lecturas_iot (
    ts              TIMESTAMPTZ NOT NULL,   -- timestamp con zona horaria
    temperatura_c   DOUBLE PRECISION NOT NULL,
    humedad_pct     DOUBLE PRECISION NOT NULL,
    fuente          TEXT DEFAULT 'counterfit'
);

-- Convierte la tabla en hypertable particionada por tiempo
SELECT create_hypertable('lecturas_iot', 'ts', if_not_exists => TRUE);
```

Una **hypertable** divide los datos en *chunks* (fragmentos) por rango de tiempo automáticamente, lo que permite consultas rápidas sobre ventanas temporales sin necesidad de índices manuales.

---

## 3. Script de captura — nodo_timescale.py

### Flujo de ejecución

1. Carga credenciales desde `.env`.
2. Abre conexión SSL a TimescaleDB Cloud.
3. Verifica / crea el esquema (`CREATE TABLE IF NOT EXISTS` + `create_hypertable`).
4. En cada ciclo (30 s por defecto):
   - Lee humedad y temperatura desde CounterFit (pin 5 y 6).
   - Inserta la fila con `ts = datetime.now(UTC)`.
   - Imprime confirmación en consola.

### Comandos

```bash
# Prueba sin CounterFit
.venv\Scripts\python actividad4/nodo_timescale.py --simulate --samples 20 --interval-sec 5

# Prueba oficial 3 horas con CounterFit en puerto 5050
.venv\Scripts\python actividad4/nodo_timescale.py --port 5050 --interval-sec 30 --samples 360
```

---

## 4. Técnicas de procesamiento de datos

Implementadas en `Procesamiento_Datos_IoT.ipynb` y reflejadas en el dashboard.

### 4.1 Preprocesamiento

| Operación | Descripción |
|-----------|-------------|
| Verificación de nulos | `df.isnull().sum()` — ninguno esperado por restricción NOT NULL |
| Duplicados | `df.duplicated().sum()` — eliminados con `drop_duplicates()` |
| Anomalías de rango | Temperatura fuera de [0, 50] °C o humedad fuera de [0, 100] % |
| Tipos de datos | `ts` convertido a `datetime64` con `pd.to_datetime()` |

### 4.2 Filtrado

```python
# Ventana temporal: últimas 24 horas
corte = datetime.now(timezone.utc) - timedelta(hours=24)
df_reciente = df[df['ts'] >= corte]

# Umbral de alerta: temperatura > 30 °C
alertas = df[df['temperatura_c'] > 30]

# Agregado horario con time_bucket() de TimescaleDB
SELECT time_bucket('1 hour', ts) AS hora,
       AVG(temperatura_c) AS temp_media,
       AVG(humedad_pct)   AS hum_media
FROM lecturas_iot
GROUP BY hora ORDER BY hora;
```

### 4.3 Transformación

```python
# Promedio móvil (ventana 5 muestras)
df['temp_movil'] = df['temperatura_c'].rolling(window=5, min_periods=1).mean()

# Normalización min-max al rango [0, 1]
df['temp_norm'] = (df['temperatura_c'] - df['temperatura_c'].min()) / \
                  (df['temperatura_c'].max() - df['temperatura_c'].min())

# Agregados diarios
df_dia = df.resample('D', on='ts').agg({'temperatura_c': ['mean','min','max'],
                                         'humedad_pct':   ['mean','min','max']})
```

---

## 5. Dashboard Streamlit

### Inicio

```bash
.venv\Scripts\streamlit run actividad4/dashboard_iot.py
```

Abre en **http://localhost:8501**

### Pestañas

| Pestaña | Descripción |
|---------|-------------|
| **Datos en vivo** | Últimas 20 lecturas, métricas actuales, gráfica de línea en tiempo real, alertas de temperatura |
| **Análisis exploratorio** | Serie temporal completa, histogramas de distribución, estadísticas descriptivas |
| **Datos procesados** | Promedio móvil superpuesto, agregados horarios con `time_bucket()`, valores normalizados |

### Controles de la barra lateral

- Intervalo de refresco automático (10–120 s)
- Ventana del promedio móvil (3–20 muestras)
- Umbral de alerta de temperatura (°C)

---

## 6. Conexión a TimescaleDB (demostración)

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Cargar credenciales desde .env en la raiz del proyecto
load_dotenv(dotenv_path=os.path.join('..', '.env'))

host     = os.environ['TS_HOST']
port     = os.environ.get('TS_PORT', '33711')
db       = os.environ.get('TS_DB', 'tsdb')
user     = os.environ.get('TS_USER', 'tsdbadmin')
password = os.environ['TS_PASSWORD']

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}",
    connect_args={"sslmode": "require"}
)

# Verificar conexion
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) AS total FROM lecturas_iot;"))
    total = result.fetchone()[0]
    print(f"Conexion exitosa. Total de lecturas almacenadas: {total}")

---

## 7. Carga y visualizacion de datos

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Cargar todos los datos
df = pd.read_sql_query(
    "SELECT ts, temperatura_c, humedad_pct FROM lecturas_iot ORDER BY ts;",
    engine,
    parse_dates=['ts']
)
print(f"Filas cargadas: {len(df)}")
print(f"Periodo: {df['ts'].min()} -> {df['ts'].max()}")
df.tail(5)

In [ ]:
# Serie temporal interactiva
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=['Temperatura (°C)', 'Humedad relativa (%)']
)

fig.add_trace(
    go.Scatter(x=df['ts'], y=df['temperatura_c'], mode='lines',
               name='Temperatura', line=dict(color='#ff7043', width=1.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df['ts'], y=df['humedad_pct'], mode='lines',
               name='Humedad', line=dict(color='#42a5f5', width=1.5)),
    row=2, col=1
)

fig.update_layout(
    title='Serie de tiempo — Lecturas IoT almacenadas en TimescaleDB',
    height=500,
    showlegend=True
)
fig.show()

---

## 8. Preprocesamiento

In [ ]:
print("=== Calidad de los datos ===")
print(f"Filas totales:       {len(df)}")
print(f"Valores nulos:       {df.isnull().sum().sum()}")
print(f"Filas duplicadas:    {df.duplicated().sum()}")

# Anomalias de rango
temp_out = df[(df['temperatura_c'] < 0) | (df['temperatura_c'] > 50)]
hum_out  = df[(df['humedad_pct'] < 0)   | (df['humedad_pct'] > 100)]
print(f"Temp fuera de rango: {len(temp_out)} filas")
print(f"Hum fuera de rango:  {len(hum_out)} filas")

print("\n=== Estadisticas descriptivas ===")
df[['temperatura_c', 'humedad_pct']].describe().round(2)

---

## 9. Filtrado

In [ ]:
from datetime import datetime, timedelta, timezone

# Filtro 1: ultimas 24 horas
corte = datetime.now(timezone.utc) - timedelta(hours=24)
df_reciente = df[df['ts'] >= corte]
print(f"Lecturas en las ultimas 24 h: {len(df_reciente)}")

# Filtro 2: alertas de temperatura alta
UMBRAL_TEMP = 30.0
alertas = df[df['temperatura_c'] > UMBRAL_TEMP]
print(f"Alertas temperatura > {UMBRAL_TEMP} C: {len(alertas)}")

# Filtro 3: agregado horario con time_bucket (SQL nativo de TimescaleDB)
df_hora = pd.read_sql_query("""
    SELECT time_bucket('1 hour', ts) AS hora,
           AVG(temperatura_c)        AS temp_media,
           AVG(humedad_pct)          AS hum_media,
           COUNT(*)                  AS n_muestras
    FROM lecturas_iot
    GROUP BY hora
    ORDER BY hora;
""", engine, parse_dates=['hora'])
print(f"\nAgregado horario: {len(df_hora)} intervalos")
df_hora.tail(5)

---

## 10. Transformacion

In [ ]:
df = df.sort_values('ts').reset_index(drop=True)

# Promedio movil (ventana = 5 muestras)
VENTANA = 5
df['temp_movil'] = df['temperatura_c'].rolling(window=VENTANA, min_periods=1).mean()
df['hum_movil']  = df['humedad_pct'].rolling(window=VENTANA, min_periods=1).mean()

# Normalizacion min-max al rango [0, 1]
def minmax(serie):
    r = serie.max() - serie.min()
    return (serie - serie.min()) / r if r > 0 else serie * 0

df['temp_norm'] = minmax(df['temperatura_c'])
df['hum_norm']  = minmax(df['humedad_pct'])

print("Columnas despues de transformacion:")
print(df.columns.tolist())
df[['ts','temperatura_c','temp_movil','temp_norm','humedad_pct','hum_norm']].tail(8)

In [ ]:
# Visualizacion: temperatura cruda vs promedio movil
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=df['ts'], y=df['temperatura_c'],
    mode='lines', name='Temperatura cruda',
    line=dict(color='#ff7043', width=1, dash='dot'),
    opacity=0.7
))
fig2.add_trace(go.Scatter(
    x=df['ts'], y=df['temp_movil'],
    mode='lines', name=f'Promedio movil (ventana={VENTANA})',
    line=dict(color='#ff1744', width=2.5)
))
fig2.update_layout(
    title='Transformacion: Temperatura cruda vs Promedio movil',
    xaxis_title='Tiempo',
    yaxis_title='Temperatura (°C)',
    height=400
)
fig2.show()

In [ ]:
# Visualizacion: variables normalizadas
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=df['ts'], y=df['temp_norm'],
    mode='lines', name='Temperatura normalizada',
    line=dict(color='#ff7043', width=1.8)
))
fig3.add_trace(go.Scatter(
    x=df['ts'], y=df['hum_norm'],
    mode='lines', name='Humedad normalizada',
    line=dict(color='#42a5f5', width=1.8)
))
fig3.update_layout(
    title='Normalizacion Min-Max — Temperatura y Humedad en escala [0, 1]',
    xaxis_title='Tiempo',
    yaxis_title='Valor normalizado',
    height=380
)
fig3.show()

---

## 11. Resumen de resultados

| Etapa | Herramienta | Resultado esperado |
|-------|-------------|-------------------|
| Captura | CounterFit + nodo_timescale.py | 360 lecturas en 3 h |
| Almacenamiento | TimescaleDB Cloud (hypertable) | Particionado automatico por tiempo |
| Preprocesamiento | pandas | 0 nulos, 0 duplicados, 0 anomalias |
| Filtrado | SQL + pandas | Ventana 24 h, alertas, agregados horarios |
| Transformacion | pandas rolling, min-max | Tendencia suavizada, escala comparativa |
| Visualizacion | Streamlit + Plotly | Dashboard interactivo en http://localhost:8501 |

---

## 12. Conclusiones

1. **TimescaleDB Cloud** resultó ser la opción óptima para almacenamiento IoT: permite escalar a millones de filas sin cambiar el código, ofrece funciones SQL nativas para series de tiempo (`time_bucket`) y es accesible desde cualquier cliente PostgreSQL estándar.

2. **El pipeline de procesamiento** (preprocesamiento → filtrado → transformación) demostró la calidad de los datos capturados: sin valores nulos ni duplicados gracias a las restricciones NOT NULL de la hypertable.

3. **El promedio móvil** suaviza efectivamente las fluctuaciones de alta frecuencia del sensor virtual, revelando tendencias térmicas que son difíciles de detectar en la señal cruda.

4. **La normalización min-max** permite comparar temperatura y humedad en la misma escala, facilitando el análisis de correlaciones entre variables.

5. **El dashboard Streamlit** cierra el ciclo IoT completo: desde el sensor físico/virtual hasta la visualización interactiva en tiempo real, constituyendo una solución desplegable en la web con mínimas modificaciones.

---

*Actividad 4 — Internet de las Cosas | Mayo 2026*